# ☕ Caso 2 — Cooperativa Café das Montanhas: estadística para decidir

> **Módulos del MBA relacionados:** Fundamentos de Estatística I, II y III

## El caso
La **Cooperativa Café das Montanhas**, del Sur de Minas Gerais, reúne 8 haciendas (*fazendas*) con 120 parcelas (*talhões*) de café arábica. Tienes los datos de las últimas **3 cosechas (*safras*): 2023, 2024 y 2025**.

El agrónomo jefe, Dr. Antônio, está cansado de decisiones tomadas por "achismo" (intuición). En la próxima asamblea los cooperados van a votar si financian **sistemas de irrigación** y un **plan de abono (*adubação*)**. Él necesita respuestas con estadística.

## Tu misión: responder estas preguntas
1. ¿Cuál es la productividad típica (sacas/ha) y cuánto varía?
2. ¿Hay datos absurdos (*outliers*)? ¿Son errores o casos reales?
3. ¿Qué variables se relacionan con la productividad: altitud, lluvia, abono, edad de la plantación?
4. ¿La irrigación **realmente** aumenta la productividad o es casualidad? (prueba de hipótesis)
5. ¿Las variedades de café producen diferente? (ANOVA)
6. ¿Cuántas sacas más da cada kg extra de abono? (regresión lineal)
7. ¿Con qué precisión conocemos la productividad media de 2025? (intervalo de confianza)

## Diccionario de datos — `cafe_safras.csv`

| columna | descripción |
|---|---|
| `safra` | año de la cosecha (2023, 2024, 2025) |
| `fazenda`, `talhao_id` | hacienda y parcela |
| `variedade` | Catuaí Vermelho, Catuaí Amarelo, Mundo Novo, Arara, Topázio |
| `altitude_m` | altitud de la parcela (m) |
| `area_ha` | área de la parcela (hectáreas) |
| `irrigado` | "Sim" / "Não" |
| `idade_lavoura_anos` | edad de las plantas (años) |
| `chuva_mm` | lluvia acumulada en el año (mm) ⚠️ hay vacíos |
| `temp_media_c` | temperatura media (°C) |
| `adubacao_kg_ha` | abono aplicado (kg de NPK por hectárea) |
| `producao_sacas` | producción total en sacas de 60 kg ⚠️ hay errores de digitación |

> En Brasil la productividad del café se mide en **sacas de 60 kg por hectárea**. La media nacional del arábica ronda las 25–30 sacas/ha. El café tiene **bienalidad**: alterna años de cosecha alta y baja.

## Qué vas a practicar
media, mediana, desvío estándar, coeficiente de variación, cuartiles · histograma y boxplot · regla del IQR · `groupby().transform()` · correlación de Pearson · prueba t (Welch) · ANOVA y Tukey · regresión lineal múltiple con `statsmodels` · intervalo de confianza · panel de gráficos con matplotlib/seaborn

> **Cómo usar este notebook:** (1) ejecuta el Paso 0; (2) en la **Parte A** sigue los comentarios; (3) compara con la **Parte B**.

## Paso 0 — Genera la base de datos

Este código simula las 3 cosechas de la cooperativa y guarda `cafe_safras.csv`. Ejecútalo una vez.

In [ ]:
# ============================================================
# PASO 0 — GENERADOR DE DATOS (ejecútalo una vez)
# ============================================================
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

fazendas = pd.DataFrame({
    "fazenda": ["Santa Clara", "Boa Vista", "Recanto", "São José",
                "Três Barras", "Alto Alegre", "Bela Vista", "Cachoeira"],
    "sigla": ["SCL", "BVI", "REC", "SJO", "TBA", "AAL", "BEL", "CAC"],
    "altitude_base": [1250, 980, 1120, 860, 1050, 1300, 920, 1180],
})
variedades = ["Catuaí Vermelho", "Catuaí Amarelo", "Mundo Novo", "Arara", "Topázio"]
efeito_variedade = {"Catuaí Vermelho": 0, "Catuaí Amarelo": -1.5, "Mundo Novo": 1.5, "Arara": 4, "Topázio": 0.5}
efeito_safra = {2023: 0, 2024: 5, 2025: -2}  # bienalidad: años de cosecha alta y baja

# 15 parcelas por hacienda, con características fijas
talhoes = []
for _, f in fazendas.iterrows():
    for k in range(1, 16):
        talhoes.append({
            "fazenda": f["fazenda"],
            "talhao_id": f"{f['sigla']}-{k:02d}",
            "variedade": rng.choice(variedades, p=[0.32, 0.18, 0.2, 0.18, 0.12]),
            "altitude_m": int(f["altitude_base"] + rng.normal(0, 45)),
            "area_ha": round(rng.uniform(2, 25), 1),
            "irrigado": rng.choice(["Sim", "Não"], p=[0.4, 0.6]),
            "idade_2023": int(rng.integers(3, 22)),
        })
talhoes = pd.DataFrame(talhoes)

linhas = []
for safra in [2023, 2024, 2025]:
    chuva_fazenda = dict(zip(fazendas["fazenda"], rng.normal(1500, 180, len(fazendas))))
    for _, t in talhoes.iterrows():
        idade = t["idade_2023"] + (safra - 2023)
        chuva = chuva_fazenda[t["fazenda"]] + rng.normal(0, 40)
        adub = rng.uniform(300, 900)
        temp = 30.5 - 0.0085 * t["altitude_m"] + rng.normal(0, 0.4)
        sacas_ha = (2 + 0.02 * adub + 0.006 * chuva + 0.005 * (t["altitude_m"] - 800)
                    + (6 if t["irrigado"] == "Sim" else 0)
                    + efeito_variedade[t["variedade"]] + efeito_safra[safra]
                    - 0.04 * (idade - 10) ** 2 + rng.normal(0, 4))
        linhas.append({
            "safra": safra, "fazenda": t["fazenda"], "talhao_id": t["talhao_id"],
            "variedade": t["variedade"], "altitude_m": t["altitude_m"], "area_ha": t["area_ha"],
            "irrigado": t["irrigado"], "idade_lavoura_anos": idade,
            "chuva_mm": float(round(chuva)), "temp_media_c": round(temp, 1),
            "adubacao_kg_ha": round(adub), "producao_sacas": round(max(sacas_ha, 3) * t["area_ha"]),
        })
cafe = pd.DataFrame(linhas)

# Problemas reales: lluvia no registrada y errores de digitación
cafe.loc[rng.choice(len(cafe), 15, replace=False), "chuva_mm"] = np.nan
erros = rng.choice(len(cafe), 3, replace=False)
cafe.loc[erros, "producao_sacas"] = cafe.loc[erros, "producao_sacas"] * 10  # alguien escribió un cero de más

cafe.to_csv("cafe_safras.csv", index=False)
print(f"✅ Archivo creado: cafe_safras.csv ({len(cafe)} filas)")

---
# ✍️ Parte A — Tu turno: el paso a paso en comentarios

Cada celda de abajo tiene **solo comentarios** con lo que debes hacer y algunas pistas.
Escribe tu código debajo de `# ✍️ Tu código aquí:` y ejecútalo.

- Si no conoces una función, búscala ("pandas como convertir texto a fecha"): buscar es parte del trabajo de un analista.
- Equivocarse es normal. Lee el mensaje de error de abajo hacia arriba: la última línea dice qué pasó.
- Si una celda que modifica datos se ejecuta dos veces, puede dar error. Vuelve a ejecutar desde el paso donde lees los archivos.

### Paso 1 — Importar las librerías

Además de pandas, usarás **matplotlib/seaborn** (gráficos estadísticos), **scipy.stats** (pruebas) y **statsmodels** (regresión).

In [ ]:
# PASO 1 — IMPORTAR LAS LIBRERÍAS
# 1. Importa pandas (pd), numpy (np), matplotlib.pyplot (plt) y seaborn (sns)
# 2. Desde scipy importa stats
# 3. Importa statsmodels.formula.api como smf (regresión con fórmulas tipo "y ~ x1 + x2")
# 4. Desde statsmodels.stats.multicomp importa pairwise_tukeyhsd
# 5. Desde matplotlib.colors importa LinearSegmentedColormap (para el mapa de calor)
# 6. Define colores fijos para usar en todo el notebook, ej.: AZUL = "#2a78d6", LARANJA = "#eb6834"

# ✍️ Tu código aquí:


### Paso 2 — Importar y conocer la base

Primer contacto con los datos: tamaño, tipos y un resumen numérico.

In [ ]:
# PASO 2 — IMPORTAR Y CONOCER LA BASE
# 1. Lee "cafe_safras.csv" en la variable cafe
# 2. Muestra .shape, .head() e .info()
# 3. Usa .describe() → mira el máximo de producao_sacas: ¿te parece normal?
# 4. ¿Cuántas parcelas hay por variedad? (.value_counts())

# ✍️ Tu código aquí:


### Paso 3 — Preparar los datos

Crea la variable que importa (productividad = sacas por hectárea) y trata los vacíos de forma inteligente.

In [ ]:
# PASO 3 — PREPARAR LOS DATOS
# 1. Crea sacas_ha = producao_sacas / area_ha (comparar producción total entre parcelas de tamaños distintos no es justo)
# 2. Cuenta los vacíos de chuva_mm
# 3. Rellénalos con la MEDIANA de lluvia de la MISMA hacienda y safra (llovió parecido ahí):
#    💡 Pista: cafe.groupby(["fazenda", "safra"])["chuva_mm"].transform("median")
# 4. Convierte fazenda, variedade e irrigado al tipo "category"
# 5. Convierte safra a texto (str): es una categoría, no un número para sumar

# ✍️ Tu código aquí:


### Paso 4 — Estadística descriptiva

Medidas de **posición** (media, mediana, cuartiles) y de **dispersión** (desvío estándar, varianza, coeficiente de variación).

In [ ]:
# PASO 4 — ESTADÍSTICA DESCRIPTIVA
# Para la columna sacas_ha calcula e imprime:
# 1. media (.mean()), mediana (.median()), desvío estándar (.std()) y varianza (.var())
# 2. coeficiente de variación = desvío / media (en %)
# 3. mínimo, máximo y cuartiles Q1, Q2, Q3 (.quantile([0.25, 0.5, 0.75]))
# 4. asimetría (.skew()) y curtosis (.kurt())
# 5. La moda tiene sentido para categorías: ¿cuál es la variedad más común? (.mode())
# 6. Tabla por variedad: cafe.groupby("variedade", observed=True)["sacas_ha"].agg([...])
# 🤔 ¿La media y la mediana están cerca? Si no, ¿qué podría estar "tirando" de la media?

# ✍️ Tu código aquí:


### Paso 5 — Distribución y outliers

Mira la forma de los datos y detecta valores extremos con la **regla del IQR**. Después decide: ¿error o dato real?

In [ ]:
# PASO 5 — DISTRIBUCIÓN Y OUTLIERS
# 1. Haz un histograma (sns.histplot) y un boxplot (sns.boxplot) de sacas_ha, lado a lado
#    💡 Pista: fig, axes = plt.subplots(1, 2, figsize=(12, 4)) y usa ax=axes[0] / ax=axes[1]
# 2. Calcula Q1, Q3 e IQR = Q3 - Q1
# 3. Límites: inferior = Q1 - 1.5*IQR | superior = Q3 + 1.5*IQR
# 4. Filtra y muestra las filas fuera de los límites (outliers)
# 5. DECIDE: un outlier no siempre es un error. Quita solo lo IMPOSIBLE (más de 100 sacas/ha)
# 6. Recalcula media y mediana sin los errores

# ✍️ Tu código aquí:


### Paso 6 — Correlación

¿Qué variables se mueven junto con la productividad? La **correlación de Pearson** (r) va de -1 a +1.

In [ ]:
# PASO 6 — CORRELACIÓN
# 1. Elige las columnas numéricas: sacas_ha, adubacao_kg_ha, chuva_mm, altitude_m,
#    temp_media_c, idade_lavoura_anos, area_ha
# 2. Calcula la matriz de correlación: cafe[columnas].corr()
# 3. Dibújala con sns.heatmap(..., annot=True, vmin=-1, vmax=1)
#    💡 Usa una escala divergente: azul (negativo) – gris (cero) – rojo (positivo)
# 4. Usa stats.pearsonr(x, y) para obtener r y el p-valor entre adubacao_kg_ha y sacas_ha
# 5. Haz un gráfico de dispersión con línea de tendencia: sns.regplot
# 🤔 altitud y temperatura tienen una correlación altísima entre sí. ¿Por qué? ¿Qué problema traería en una regresión?

# ✍️ Tu código aquí:


### Paso 7 — Prueba de hipótesis: ¿la irrigación funciona?

La **prueba t** compara las medias de dos grupos y responde: ¿la diferencia es real o pudo ser casualidad?

In [ ]:
# PASO 7 — PRUEBA DE HIPÓTESIS: ¿LA IRRIGACIÓN FUNCIONA?
# 1. Separa sacas_ha en dos grupos: irrigado == "Sim" y irrigado == "Não"
# 2. Calcula la media de cada grupo y la diferencia
# 3. Escribe las hipótesis (en un comentario):
#    H0: las medias son iguales | H1: las medias son diferentes
# 4. Aplica stats.ttest_ind(grupo1, grupo2, equal_var=False)  ← prueba de Welch
# 5. Si p-valor < 0.05 → rechazas H0 (la diferencia es estadísticamente significativa)
# 6. Haz un boxplot de sacas_ha por irrigado

# ✍️ Tu código aquí:


### Paso 8 — ANOVA: ¿las variedades producen diferente?

Con **más de dos grupos** no se hacen muchas pruebas t: se usa **ANOVA**. Si da significativo, la prueba de **Tukey** dice qué pares difieren.

In [ ]:
# PASO 8 — ANOVA: ¿LAS VARIEDADES PRODUCEN DIFERENTE?
# 1. Crea una lista con los valores de sacas_ha de cada variedad
#    💡 Pista: [g["sacas_ha"] for _, g in cafe.groupby("variedade", observed=True)]
# 2. Aplica stats.f_oneway(*grupos) → F y p-valor
# 3. Si p < 0.05, aplica pairwise_tukeyhsd(cafe["sacas_ha"], cafe["variedade"].astype(str))
#    y muestra .summary() → la columna "reject" = True indica que ese par es diferente
# 4. Haz un boxplot por variedad, ordenado por la media

# ✍️ Tu código aquí:


### Paso 9 — Regresión lineal múltiple

La regresión estima **cuánto** cambia la productividad por cada unidad de cada variable, manteniendo las otras constantes.

In [ ]:
# PASO 9 — REGRESIÓN LINEAL MÚLTIPLE
# 1. Ajusta el modelo con smf.ols("sacas_ha ~ adubacao_kg_ha + chuva_mm + altitude_m + C(irrigado) + C(safra)", data=cafe).fit()
#    C(...) indica variable categórica (se convierte en "dummies" automáticamente)
# 2. Muestra modelo.summary() y busca: R², coeficientes (coef) y p-valores (P>|t|)
# 3. Interpreta en palabras:
#    - ¿cuántas sacas/ha más por cada +100 kg/ha de abono?
#    - ¿cuántas por cada +100 mm de lluvia?
#    - ¿cuánto suma irrigar?
# 4. Predice la productividad de una parcela nueva con modelo.predict(DataFrame con los valores)

# ✍️ Tu código aquí:


### Paso 10 — Intervalo de confianza

Con una muestra nunca sabemos la media verdadera. El **intervalo de confianza del 95%** da un rango donde probablemente está.

In [ ]:
# PASO 10 — INTERVALO DE CONFIANZA
# 1. Filtra sacas_ha de la safra "2025"
# 2. Calcula la media y el error estándar (stats.sem)
# 3. Calcula el IC 95%: stats.t.interval(0.95, df=n-1, loc=media, scale=error_estandar)
# 4. Repite para cada safra en una tabla. ¿Ves la bienalidad (año alto / año bajo)?

# ✍️ Tu código aquí:


### Paso 11 — Panel visual para el agrónomo

Resume los hallazgos en **un panel de 4 gráficos** listo para mostrar en la asamblea, y guárdalo como imagen.

In [ ]:
# PASO 11 — PANEL VISUAL PARA EL AGRÓNOMO
# 1. Crea una figura 2x2: fig, ax = plt.subplots(2, 2, figsize=(13, 9))
# 2. ax[0, 0]: media por safra con barras de error (IC 95%) → muestra la bienalidad
# 3. ax[0, 1]: productividad media por hacienda (barras horizontales ordenadas)
# 4. ax[1, 0]: boxplot irrigado vs. no irrigado
# 5. ax[1, 1]: dispersión abono × productividad coloreada por irrigado
# 6. Título general con fig.suptitle, plt.tight_layout() y guarda con plt.savefig("reporte_cafe.png", dpi=150)

# ✍️ Tu código aquí:


---
# 🔑 Parte B — Solución comentada

**Intenta resolver la Parte A antes de mirar esto.** Después compara: no hay una única forma correcta,
pero fíjate en las funciones que usé y en los comentarios que explican el porqué.

> Ejecuta la Parte B **en orden, desde el Paso 1**: vuelve a leer los archivos originales, así que no depende de lo que hiciste en la Parte A.

### ✅ Paso 1 — Importar las librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from matplotlib.colors import LinearSegmentedColormap

# Colores fijos: "Sim" (irrigado) siempre azul y "Não" siempre naranja, en todos los gráficos
AZUL, LARANJA = "#2a78d6", "#eb6834"
CORES_IRRIGADO = {"Sim": AZUL, "Não": LARANJA}
sns.set_theme(style="whitegrid")
plt.rcParams.update({"axes.facecolor": "#fcfcfb", "figure.facecolor": "#fcfcfb",
                     "grid.color": "#e1e0d9", "axes.edgecolor": "#c3c2b7"})

### ✅ Paso 2 — Importar y conocer la base

In [ ]:
cafe = pd.read_csv("cafe_safras.csv")
print(cafe.shape)
display(cafe.head())
cafe.info()
display(cafe.describe().round(1))
print(cafe["variedade"].value_counts())

### ✅ Paso 3 — Preparar los datos

In [ ]:
# 3.1 Variable objetivo: productividad
cafe["sacas_ha"] = cafe["producao_sacas"] / cafe["area_ha"]

# 3.2 Vacíos de lluvia → mediana de la misma hacienda y safra
print("Vacíos antes:", cafe["chuva_mm"].isna().sum())
mediana_grupo = cafe.groupby(["fazenda", "safra"])["chuva_mm"].transform("median")
cafe["chuva_mm"] = cafe["chuva_mm"].fillna(mediana_grupo)
print("Vacíos después:", cafe["chuva_mm"].isna().sum())

# 3.3 Tipos
for col in ["fazenda", "variedade", "irrigado"]:
    cafe[col] = cafe[col].astype("category")
cafe["safra"] = cafe["safra"].astype(str)

💡 **¿Por qué no rellenar con la media general?** Porque la lluvia depende del lugar y del año. `transform` calcula la mediana de cada grupo y la devuelve "alineada" con cada fila: es una de las funciones más útiles de pandas.

### ✅ Paso 4 — Estadística descriptiva

In [ ]:
x = cafe["sacas_ha"]
print(f"Media:               {x.mean():.2f} sacas/ha")
print(f"Mediana:             {x.median():.2f}")
print(f"Desvío estándar:     {x.std():.2f}")
print(f"Varianza:            {x.var():.2f}")
print(f"Coef. de variación:  {x.std() / x.mean():.1%}")
print(f"Mín / Máx:           {x.min():.2f} / {x.max():.2f}")
print(f"Q1, Q2, Q3:          {x.quantile([0.25, 0.5, 0.75]).round(2).tolist()}")
print(f"Asimetría:           {x.skew():.2f}")
print(f"Curtosis:            {x.kurt():.2f}")
print(f"Variedad más común:  {cafe['variedade'].mode()[0]}")

display(cafe.groupby("variedade", observed=True)["sacas_ha"]
        .agg(["count", "mean", "median", "std", "min", "max"]).round(2))

💡 **Qué observar:** el máximo es absurdo (cientos de sacas/ha) y la asimetría y la curtosis son enormes. Tres valores extremos inflan la media y el desvío estándar; la mediana casi no se mueve. Por eso la mediana es una medida **robusta**.

### ✅ Paso 5 — Distribución y outliers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(cafe["sacas_ha"], bins=40, color=AZUL, ax=axes[0])
axes[0].set_title("Histograma de productividad (sacas/ha)")
sns.boxplot(x=cafe["sacas_ha"], color=AZUL, ax=axes[1])
axes[1].set_title("Boxplot: los puntos sueltos son outliers")
plt.tight_layout()
plt.show()

# Regla del IQR
q1, q3 = cafe["sacas_ha"].quantile([0.25, 0.75])
iqr = q3 - q1
lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = cafe[(cafe["sacas_ha"] < lim_inf) | (cafe["sacas_ha"] > lim_sup)]
print(f"Límites IQR: {lim_inf:.1f} a {lim_sup:.1f} sacas/ha → {len(outliers)} outliers")
display(outliers[["safra", "fazenda", "talhao_id", "area_ha", "producao_sacas", "sacas_ha"]].round(1))

# Decisión: más de 100 sacas/ha es imposible → error de digitación → se elimina.
# Los otros outliers son raros pero posibles → se quedan.
cafe = cafe[cafe["sacas_ha"] < 100].copy()
print(f"Sin errores → media: {cafe['sacas_ha'].mean():.2f} | mediana: {cafe['sacas_ha'].median():.2f} "
      f"| asimetría: {cafe['sacas_ha'].skew():.2f}")

sns.histplot(cafe["sacas_ha"], bins=30, color=AZUL, kde=True)
plt.title("Distribución sin los errores: casi simétrica")
plt.show()

💡 **Regla de oro:** outlier ≠ error. La regla del IQR solo **señala** candidatos; quien decide eres tú, con conocimiento del negocio. Borrar datos reales que "molestan" es una de las formas más comunes de engañarse con estadística.

### ✅ Paso 6 — Correlación

In [ ]:
num = ["sacas_ha", "adubacao_kg_ha", "chuva_mm", "altitude_m",
       "temp_media_c", "idade_lavoura_anos", "area_ha"]
corr = cafe[num].corr(method="pearson")

divergente = LinearSegmentedColormap.from_list("divergente", ["#1c5cab", "#f0efec", "#e34948"])
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap=divergente, vmin=-1, vmax=1,
            linewidths=2, linecolor="#fcfcfb")
plt.title("Matriz de correlación (Pearson)")
plt.show()

r, p = stats.pearsonr(cafe["adubacao_kg_ha"], cafe["sacas_ha"])
print(f"Abono × productividad: r = {r:.2f} | p-valor = {p:.2g}")

sns.regplot(data=cafe, x="adubacao_kg_ha", y="sacas_ha",
            scatter_kws={"alpha": 0.4, "color": AZUL}, line_kws={"color": LARANJA, "linewidth": 2})
plt.title("Más abono → más sacas/ha")
plt.show()

💡 **Correlación no es causalidad.** Altitud y temperatura están casi perfectamente correlacionadas (cuanto más alto, más frío). Si pones las dos en una regresión, el modelo no sabe a cuál atribuir el efecto: eso se llama **multicolinealidad**. Por eso en el Paso 9 usamos solo la altitud.

### ✅ Paso 7 — Prueba de hipótesis: ¿la irrigación funciona?

In [ ]:
sim = cafe.loc[cafe["irrigado"] == "Sim", "sacas_ha"]
nao = cafe.loc[cafe["irrigado"] == "Não", "sacas_ha"]
print(f"Media irrigado: {sim.mean():.2f} | no irrigado: {nao.mean():.2f} "
      f"| diferencia: {sim.mean() - nao.mean():+.2f} sacas/ha")

# H0: medias iguales | H1: medias diferentes. Welch no supone varianzas iguales (más seguro).
t, p = stats.ttest_ind(sim, nao, equal_var=False)
print(f"t = {t:.2f} | p-valor = {p:.2g}")
if p < 0.05:
    print("➡️ Rechazamos H0: la irrigación SÍ cambia la productividad.")
else:
    print("➡️ No hay evidencia suficiente de diferencia.")

sns.boxplot(data=cafe, x="irrigado", y="sacas_ha", hue="irrigado",
            palette=CORES_IRRIGADO, order=["Sim", "Não"], legend=False)
plt.title("Productividad: irrigado vs. no irrigado")
plt.show()

💡 **Cómo leer el p-valor:** es la probabilidad de ver una diferencia así de grande **si en realidad no hubiera ninguna**. Un p-valor minúsculo dice que la casualidad casi no explica la diferencia. Ojo: "significativo" no quiere decir "grande". Para saber si **compensa** invertir, mira el tamaño de la diferencia (sacas/ha) y compáralo con el costo de irrigar.

### ✅ Paso 8 — ANOVA: ¿las variedades producen diferente?

In [ ]:
grupos = [g["sacas_ha"] for _, g in cafe.groupby("variedade", observed=True)]
f, p = stats.f_oneway(*grupos)
print(f"ANOVA: F = {f:.2f} | p-valor = {p:.2g}")

tukey = pairwise_tukeyhsd(cafe["sacas_ha"], cafe["variedade"].astype(str), alpha=0.05)
print(tukey.summary())

orden = cafe.groupby("variedade", observed=True)["sacas_ha"].mean().sort_values().index
plt.figure(figsize=(9, 4))
sns.boxplot(data=cafe, x="variedade", y="sacas_ha", order=orden, color=AZUL)
plt.title("Productividad por variedad (ordenada por la media)")
plt.show()

### ✅ Paso 9 — Regresión lineal múltiple

In [ ]:
modelo = smf.ols("sacas_ha ~ adubacao_kg_ha + chuva_mm + altitude_m + C(irrigado) + C(safra)",
                 data=cafe).fit()
print(modelo.summary())

coef = modelo.params
print(f"\nR² = {modelo.rsquared:.2f} → el modelo explica {modelo.rsquared:.0%} de la variación de la productividad")
print(f"Cada +100 kg/ha de abono  → {100 * coef['adubacao_kg_ha']:+.1f} sacas/ha")
print(f"Cada +100 mm de lluvia    → {100 * coef['chuva_mm']:+.1f} sacas/ha")
print(f"Cada +100 m de altitud    → {100 * coef['altitude_m']:+.1f} sacas/ha")
print(f"Irrigar                   → {coef['C(irrigado)[T.Sim]']:+.1f} sacas/ha")

# Predicción para una parcela nueva
nova = pd.DataFrame({"adubacao_kg_ha": [700], "chuva_mm": [1400], "altitude_m": [1100],
                     "irrigado": ["Sim"], "safra": ["2025"]})
print(f"\nPredicción para la parcela nueva: {modelo.predict(nova).iloc[0]:.1f} sacas/ha")

💡 **Cómo leer el resumen:** `coef` = efecto de cada variable; `P>|t|` < 0.05 = efecto estadísticamente significativo; `R-squared` = cuánto de la variación explica el modelo. El resto (lo no explicado) viene de factores que no medimos: suelo, plagas, manejo, azar.

### ✅ Paso 10 — Intervalo de confianza

In [ ]:
s25 = cafe.loc[cafe["safra"] == "2025", "sacas_ha"]
media, ep = s25.mean(), stats.sem(s25)
li, ls = stats.t.interval(0.95, df=len(s25) - 1, loc=media, scale=ep)
print(f"Safra 2025: media {media:.1f} sacas/ha | IC 95%: [{li:.1f}; {ls:.1f}]")

ic = cafe.groupby("safra")["sacas_ha"].agg(n="count", media="mean", ep=stats.sem)
ic["ic_inf"] = ic["media"] - stats.t.ppf(0.975, ic["n"] - 1) * ic["ep"]
ic["ic_sup"] = ic["media"] + stats.t.ppf(0.975, ic["n"] - 1) * ic["ep"]
display(ic.round(2))

### ✅ Paso 11 — Panel visual para el agrónomo

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 9))

# (a) Bienalidad: media por safra ± IC 95%
erro = ic["ic_sup"] - ic["media"]
ax[0, 0].bar(ic.index, ic["media"], yerr=erro, color=AZUL, capsize=5, width=0.5)
ax[0, 0].set_title("Media por safra (± IC 95%): bienalidad")
ax[0, 0].set_ylabel("sacas/ha")

# (b) Por hacienda
por_faz = cafe.groupby("fazenda", observed=True)["sacas_ha"].mean().sort_values()
ax[0, 1].barh(por_faz.index.astype(str), por_faz.values, color=AZUL, height=0.6)
ax[0, 1].set_title("Productividad media por hacienda")
ax[0, 1].set_xlabel("sacas/ha")

# (c) Irrigación
sns.boxplot(data=cafe, x="irrigado", y="sacas_ha", hue="irrigado", palette=CORES_IRRIGADO,
            order=["Sim", "Não"], legend=False, ax=ax[1, 0])
ax[1, 0].set_title(f"Irrigado produce {sim.mean() - nao.mean():+.1f} sacas/ha (p < 0.05)")

# (d) Abono × productividad
sns.scatterplot(data=cafe, x="adubacao_kg_ha", y="sacas_ha", hue="irrigado",
                hue_order=["Sim", "Não"], palette=CORES_IRRIGADO, alpha=0.6, ax=ax[1, 1])
ax[1, 1].set_title("Abono × productividad")
ax[1, 1].legend(title="Irrigado", frameon=False)

fig.suptitle("Cooperativa Café das Montanhas · Reporte de productividad 2023–2025",
             fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("reporte_cafe.png", dpi=150)
plt.show()
print("✅ Imagen guardada: reporte_cafe.png")

---
## 🎤 Conclusiones para el Dr. Antônio (con tus números)
1. **Productividad típica:** unas 26,5 sacas/ha. Una vez quitados los errores, la media y la mediana casi coinciden y la distribución es simétrica.
2. **Datos:** había 3 errores de digitación (producción ×10). Con ellos la media sube a 28,5 y el desvío estándar se triplica. El IQR también marcó 2 parcelas con productividad muy baja, que son reales y se quedan en la base.
3. **Irrigación:** las parcelas irrigadas producen unas 6 sacas/ha más (p-valor minúsculo). Ahora la asamblea tiene que comparar esa ganancia con el costo del sistema.
4. **Abono:** cada 100 kg/ha extra de NPK suma unas 2 sacas/ha, manteniendo constantes las otras variables. El modelo explica cerca del 50% de la variación.
5. **Bienalidad:** 2024 fue año alto; al planear el flujo de caja, no hay que tomar un año aislado como referencia.

## 🚀 Retos extra
1. Agrega `C(variedade)` e `idade_lavoura_anos` a la regresión. ¿Sube el R²? ¿Qué pasa si agregas `I(idade_lavoura_anos**2)`? (pista: la productividad del café es máxima en plantas de edad media)
2. Verifica la normalidad de los residuos del modelo con `stats.shapiro(modelo.resid)` y un histograma.
3. Calcula el **valor económico**: si la saca vale R\$ 1.500 y el abono cuesta R\$ 4/kg, ¿cuántos kg/ha conviene aplicar?
4. Haz la prueba t comparando 2024 vs. 2025 **solo para las mismas parcelas** (prueba t pareada: `stats.ttest_rel`).